## AD and TP level

In [200]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 配置路径
# ==========================================
base_path = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC"
file_label = os.path.join(base_path, "Part 5 other figures in Fig1-4 Fig S1-6/anderson_adt.txt") # 请确认这个文件是否包含 0,1,2
file_cluster = os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/mydata_pheno02.csv")
output_prefix = os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/Results_Merged_Fisher")
df_label = pd.read_csv(file_label, sep='\t', engine='python')
df_label.columns = [str(c).strip().lower() for c in df_label.columns]
col_group = df_label.columns[0] 
col_sample = df_label.columns[1]

df_label.rename(columns={col_group: 'Group_Raw', col_sample: 'Sample_Name_Raw'}, inplace=True)
df_label['Group_Num'] = pd.to_numeric(df_label['Group_Raw'], errors='ignore')
df_c_raw = pd.read_csv(file_cluster)


In [201]:
import pandas as pd
import re

# ---------------------------------------------------------
# 1. 数据预处理函数
# ---------------------------------------------------------
def clean_sample_name(name):
    """
    统一格式规则：
    1. 去除末尾的 '.tsv'
    2. 去除开头的 'm' (将 mADT-1 变为 ADT-1)
    3. 去除首尾空格
    """
    if not isinstance(name, str):
        return name
    
    # 去除 .tsv
    cleaned = re.sub(r'\.tsv$', '', name.strip())
    
    # 去除开头的 'm' (仅当后面紧跟 ADT 时，防止误删其他以 m 开头的名字，虽然这里主要是 mADT)
    # 使用正则 ^m(?=ADT) 表示：如果开头是 m 且后面紧跟着 ADT，则删除 m
    cleaned = re.sub(r'^m(?=ADT)', '', cleaned)
    
    return cleaned


df_label['Match_Key'] = df_label['Sample_Name_Raw'].apply(clean_sample_name)


if 'Class' in df_c_raw.columns:
    source_series = df_c_raw['Class']
else:
    # 如果 Class 是索引
    source_series = df_c_raw.index

df_c_raw['Match_Key'] = source_series.apply(clean_sample_name)

df_merged = pd.merge(
    df_label, 
    df_c_raw, 
    on='Match_Key', 
    how='inner',
    suffixes=('_label', '_c_raw') # 给重复列名加后缀，避免冲突
)

df_final = df_merged.rename(columns={'Match_Key': 'Unified_Sample_Name'})


print("合并成功！重叠样本数量:", len(df_final))
print("\n前 10 行预览:")
print(df_final[['Unified_Sample_Name', 'Sample_Name_Raw', 'Class']].head(10))



合并成功！重叠样本数量: 50

前 10 行预览:
  Unified_Sample_Name Sample_Name_Raw        Class
0               ADT-1           ADT-1   mADT-1.tsv
1               ADT-2           ADT-2   mADT-2.tsv
2               ADT-3           ADT-3   mADT-3.tsv
3               ADT-4           ADT-4   mADT-4.tsv
4               ADT-5           ADT-5   mADT-5.tsv
5               ADT-6           ADT-6   mADT-6.tsv
6               ADT-7           ADT-7   mADT-7.tsv
7               ADT-8           ADT-8   mADT-8.tsv
8               ADT-9           ADT-9   mADT-9.tsv
9              ADT-10          ADT-10  mADT-10.tsv


In [202]:
# 定义映射关系
mapping = {0: 0, 1: 0, 2: 1}

# 应用映射
df_final['Group_Num'] = df_final['Group_Num'].map(mapping)

# 检查是否有未匹配到的值 (如果有 NaN 产生，说明原数据中有 0,1,2 以外的值)
if df_final['Group_Num'].isnull().any():
    print("警告：发现未定义的原始值，已转换为 NaN")
else:
    print("转换成功！")

# 查看转换后的分布
print(df_final['Group_Num'].value_counts().sort_index())

转换成功！
0    30
1    20
Name: Group_Num, dtype: int64


In [203]:
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np

# ---------------------------------------------------------
# 1. 构建列联表 (Crosstab)
# ---------------------------------------------------------
# 确保数据是整数或类别类型，去除可能的 NaN
df_clean = df_final.dropna(subset=['Group_Num', 'TumorPurityLevel'])

# 创建交叉表
contingency_table = pd.crosstab(df_clean['Group_Num'], df_clean['TumorPurityLevel'])

print("列联表 (Crosstab):")
print(contingency_table)
print("-" * 30)

row_sum = contingency_table.sum(axis=1)
col_sum = contingency_table.sum(axis=0)
total = contingency_table.values.sum()

expected = np.outer(row_sum, col_sum) / total
expected = pd.DataFrame(expected,
                        index=contingency_table.index,
                        columns=contingency_table.columns)
p_table = pd.DataFrame(index=contingency_table.index,
                       columns=contingency_table.columns)

for i in contingency_table.index:
    for j in contingency_table.columns:
        
        a = contingency_table.loc[i, j]
        b = contingency_table.loc[i].sum() - a
        c = contingency_table[j].sum() - a
        d = total - (a + b + c)
        
        table = [[a, b],
                 [c, d]]
        
        _, p = fisher_exact(table,alternative="greater")
        p_table.loc[i, j] = p

p_table = p_table.astype(float)
p_table.to_csv("TL_AD.csv")

列联表 (Crosstab):
TumorPurityLevel  H   L   M
Group_Num                  
0                 1  20   9
1                 3   6  11
------------------------------


## tratment and Tumtrpurity

In [204]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 配置路径
# ==========================================
base_path = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC"
# = os.path.join(base_path, "Part 5 other figures in Fig1-4 Fig S1-6/anderson_adt.txt") # 请确认这个文件是否包含 0,1,2
file_cluster = os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/mydata_pheno02.csv")
output_prefix = os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/Results_Merged_Fisher")
#df_label = pd.read_csv(file_label, sep='\t', engine='python')
#df_label.columns = [str(c).strip().lower() for c in df_label.columns]
#df_label.rename(columns={col_group: 'Group_Raw', col_sample: 'Sample_Name_Raw'}, inplace=True)
#df_label['Group_Num'] = pd.to_numeric(df_label['Group_Raw'], errors='ignore')
df_c_raw = pd.read_csv(file_cluster)
import numpy as np

# 确保 'Class' 列是字符串类型，防止报错
df_c_raw['Class'] = df_c_raw['Class'].astype(str)

# 逻辑：如果以 'mADT' 开头则为 0，否则 (即 'N' 开头) 为 1
df_c_raw['Group_Num'] = np.where(df_c_raw['Class'].str.startswith('mADT'), 0, 1)

print("新建列成功！前 5 行预览：")
print(df_c_raw[['Class', 'Group_Num']].head())


新建列成功！前 5 行预览：
                   Class  Group_Num
mADT-1.tsv    mADT-1.tsv          0
mADT-10.tsv  mADT-10.tsv          0
mADT-11.tsv  mADT-11.tsv          0
mADT-12.tsv  mADT-12.tsv          0
mADT-13.tsv  mADT-16.tsv          0


In [205]:
df_final=df_c_raw.copy()
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np

# ---------------------------------------------------------
# 1. 构建列联表 (Crosstab)
# ---------------------------------------------------------
# 确保数据是整数或类别类型，去除可能的 NaN
df_clean = df_final.dropna(subset=['Group_Num', 'TumorPurityLevel'])

# 创建交叉表
contingency_table = pd.crosstab(df_clean['Group_Num'], df_clean['TumorPurityLevel'])

print("列联表 (Crosstab):")
print(contingency_table)
print("-" * 30)

row_sum = contingency_table.sum(axis=1)
col_sum = contingency_table.sum(axis=0)
total = contingency_table.values.sum()

expected = np.outer(row_sum, col_sum) / total
expected = pd.DataFrame(expected,
                        index=contingency_table.index,
                        columns=contingency_table.columns)
p_table = pd.DataFrame(index=contingency_table.index,
                       columns=contingency_table.columns)

for i in contingency_table.index:
    for j in contingency_table.columns:
        
        a = contingency_table.loc[i, j]
        b = contingency_table.loc[i].sum() - a
        c = contingency_table[j].sum() - a
        d = total - (a + b + c)
        
        table = [[a, b],
                 [c, d]]
        
        _, p = fisher_exact(table,alternative="greater")
        p_table.loc[i, j] = p

p_table = p_table.astype(float)
p_table.to_csv("TL_Treat.csv")

列联表 (Crosstab):
TumorPurityLevel   H   L   M
Group_Num                   
0                  4  26  20
1                 20  10  26
------------------------------


## TP and MD

In [206]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 配置路径
# ==========================================
base_path = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC"
file_label = os.path.join(base_path, "Part 5 other figures in Fig1-4 Fig S1-6/anderson_adt.txt") # 请确认这个文件是否包含 0,1,2
file_cluster = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC/Part 3 discovery cohort TP SP TPL/mydata_pheno_anno.csv"
output_prefix = os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/Results_Merged_Fisher")
df_label = pd.read_csv(file_label, sep='\t', engine='python')
df_label.columns = [str(c).strip().lower() for c in df_label.columns]
col_group = df_label.columns[0] 
col_sample = df_label.columns[1]
df_label.rename(columns={col_group: 'Group_Raw', col_sample: 'Sample_Name_Raw'}, inplace=True)
df_label['Group_Num'] = pd.to_numeric(df_label['Group_Raw'], errors='ignore')
df_c_raw = pd.read_csv(file_cluster)

In [207]:
import pandas as pd
import re

# ---------------------------------------------------------
# 1. 数据预处理函数
# ---------------------------------------------------------
def clean_sample_name(name):
    """
    统一格式规则：
    1. 去除末尾的 '.tsv'
    2. 去除开头的 'm' (将 mADT-1 变为 ADT-1)
    3. 去除首尾空格
    """
    if not isinstance(name, str):
        return name
    
    # 去除 .tsv
    cleaned = re.sub(r'\.tsv$', '', name.strip())
    
    # 去除开头的 'm' (仅当后面紧跟 ADT 时，防止误删其他以 m 开头的名字，虽然这里主要是 mADT)
    # 使用正则 ^m(?=ADT) 表示：如果开头是 m 且后面紧跟着 ADT，则删除 m
    cleaned = re.sub(r'^m(?=ADT)', '', cleaned)
    
    return cleaned


df_label['Match_Key'] = df_label['Sample_Name_Raw'].apply(clean_sample_name)


if 'Class' in df_c_raw.columns:
    source_series = df_c_raw['Class']
else:
    # 如果 Class 是索引
    source_series = df_c_raw.index

df_c_raw['Match_Key'] = source_series.apply(clean_sample_name)

df_merged = pd.merge(
    df_label, 
    df_c_raw, 
    on='Match_Key', 
    how='inner',
    suffixes=('_label', '_c_raw') # 给重复列名加后缀，避免冲突
)

df_final = df_merged.rename(columns={'Match_Key': 'Unified_Sample_Name'})


print("合并成功！重叠样本数量:", len(df_final))
print("\n前 10 行预览:")
print(df_final[['Unified_Sample_Name', 'Sample_Name_Raw', 'Class']].head(10))



合并成功！重叠样本数量: 50

前 10 行预览:
  Unified_Sample_Name Sample_Name_Raw        Class
0               ADT-1           ADT-1   mADT-1.tsv
1               ADT-2           ADT-2   mADT-2.tsv
2               ADT-3           ADT-3   mADT-3.tsv
3               ADT-4           ADT-4   mADT-4.tsv
4               ADT-5           ADT-5   mADT-5.tsv
5               ADT-6           ADT-6   mADT-6.tsv
6               ADT-7           ADT-7   mADT-7.tsv
7               ADT-8           ADT-8   mADT-8.tsv
8               ADT-9           ADT-9   mADT-9.tsv
9              ADT-10          ADT-10  mADT-10.tsv


In [208]:
# 定义映射关系
mapping = {0: 0, 1: 0, 2: 1}

# 应用映射
df_final['Group_Num'] = df_final['Group_Num'].map(mapping)

# 检查是否有未匹配到的值 (如果有 NaN 产生，说明原数据中有 0,1,2 以外的值)
if df_final['Group_Num'].isnull().any():
    print("警告：发现未定义的原始值，已转换为 NaN")
else:
    print("转换成功！")

# 查看转换后的分布
print(df_final['Group_Num'].value_counts().sort_index())

转换成功！
0    30
1    20
Name: Group_Num, dtype: int64


In [209]:
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np

# ---------------------------------------------------------
# 1. 构建列联表 (Crosstab)
# ---------------------------------------------------------
# 确保数据是整数或类别类型，去除可能的 NaN
df_clean = df_final.dropna(subset=['Group_Num', 'Cluster'])

# 创建交叉表
contingency_table = pd.crosstab(df_clean['Group_Num'], df_clean['Cluster'])

print("列联表 (Crosstab):")
print(contingency_table)
print("-" * 30)

row_sum = contingency_table.sum(axis=1)
col_sum = contingency_table.sum(axis=0)
total = contingency_table.values.sum()

expected = np.outer(row_sum, col_sum) / total
expected = pd.DataFrame(expected,
                        index=contingency_table.index,
                        columns=contingency_table.columns)
p_table = pd.DataFrame(index=contingency_table.index,
                       columns=contingency_table.columns)

for i in contingency_table.index:
    for j in contingency_table.columns:
        
        a = contingency_table.loc[i, j]
        b = contingency_table.loc[i].sum() - a
        c = contingency_table[j].sum() - a
        d = total - (a + b + c)
        
        table = [[a, b],
                 [c, d]]
        
        _, p = fisher_exact(table,alternative="greater")
        p_table.loc[i, j] = p

p_table = p_table.astype(float)
p_table.to_csv("TP_AD.csv")

列联表 (Crosstab):
Cluster    TP_1  TP_10  TP_11  TP_12  TP_2  TP_3  TP_4-13  TP_5  TP_6  TP_7  \
Group_Num                                                                     
0             2      1      1      0     7     7        1     1     2     1   
1             1      0      0      1     5     3        1     2     1     4   

Cluster    TP_8  TP_9  
Group_Num              
0             4     3  
1             0     2  
------------------------------


## TP and treat

In [210]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 配置路径
# ==========================================
base_path = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC"
file_label = os.path.join(base_path, "Part 5 other figures in Fig1-4 Fig S1-6/anderson_adt.txt") # 请确认这个文件是否包含 0,1,2
file_cluster = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC/Part 3 discovery cohort TP SP TPL/mydata_pheno_anno.csv"
output_prefix = os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/Results_Merged_Fisher")
df_label = pd.read_csv(file_label, sep='\t', engine='python')
df_label.columns = [str(c).strip().lower() for c in df_label.columns]
col_group = df_label.columns[0] 
col_sample = df_label.columns[1]
df_label.rename(columns={col_group: 'Group_Raw', col_sample: 'Sample_Name_Raw'}, inplace=True)
df_label['Group_Num'] = pd.to_numeric(df_label['Group_Raw'], errors='ignore')
df_c_raw = pd.read_csv(file_cluster)

In [211]:
import numpy as np

# 确保 'Class' 列是字符串类型，防止报错
df_c_raw['Class'] = df_c_raw['Class'].astype(str)

# 逻辑：如果以 'mADT' 开头则为 0，否则 (即 'N' 开头) 为 1
df_c_raw['Group_Num'] = np.where(df_c_raw['Class'].str.startswith('mADT'), 0, 1)

print("新建列成功！前 5 行预览：")
print(df_c_raw[['Class', 'Group_Num']].head())

新建列成功！前 5 行预览：
         Class  Group_Num
0   mADT-1.tsv          0
1  mADT-10.tsv          0
2  mADT-11.tsv          0
3  mADT-12.tsv          0
4  mADT-13.tsv          0


In [212]:
df_final=df_c_raw.copy()
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np

# ---------------------------------------------------------
# 1. 构建列联表 (Crosstab)
# ---------------------------------------------------------
# 确保数据是整数或类别类型，去除可能的 NaN
df_clean = df_final.dropna(subset=['Group_Num', 'Cluster'])

# 创建交叉表
contingency_table = pd.crosstab(df_clean['Group_Num'], df_clean['Cluster'])

print("列联表 (Crosstab):")
print(contingency_table)
print("-" * 30)

row_sum = contingency_table.sum(axis=1)
col_sum = contingency_table.sum(axis=0)
total = contingency_table.values.sum()

expected = np.outer(row_sum, col_sum) / total
expected = pd.DataFrame(expected,
                        index=contingency_table.index,
                        columns=contingency_table.columns)
p_table = pd.DataFrame(index=contingency_table.index,
                       columns=contingency_table.columns)

for i in contingency_table.index:
    for j in contingency_table.columns:
        
        a = contingency_table.loc[i, j]
        b = contingency_table.loc[i].sum() - a
        c = contingency_table[j].sum() - a
        d = total - (a + b + c)
        
        table = [[a, b],
                 [c, d]]
        
        _, p = fisher_exact(table,alternative="greater")
        p_table.loc[i, j] = p

p_table = p_table.astype(float)
p_table.to_csv("TP_Treat.csv")

列联表 (Crosstab):
Cluster    TP_1  TP_10  TP_11  TP_12  TP_14  TP_15  TP_16  TP_17  TP_2  TP_3  \
Group_Num                                                                      
0             3      1      1      1      0      0      0      0    12    10   
1             0      1      0     19      6      1      2      1     0     0   

Cluster    TP_4-13  TP_5  TP_6  TP_7  TP_8  TP_9  
Group_Num                                         
0                2     3     3     5     4     5  
1               16     6     3     0     0     1  
------------------------------


## SP and MD

In [213]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 配置路径
# ==========================================
base_path = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC"
file_label = os.path.join(base_path, "Part 5 other figures in Fig1-4 Fig S1-6/anderson_adt.txt") # 请确认这个文件是否包含 0,1,2
file_cluster = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC/Part 3 discovery cohort TP SP TPL/dat_CN_percent_class.csv"
output_prefix = os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/Results_Merged_Fisher")
df_label = pd.read_csv(file_label, sep='\t', engine='python')
df_label.columns = [str(c).strip().lower() for c in df_label.columns]
col_group = df_label.columns[0] 
col_sample = df_label.columns[1]
df_label.rename(columns={col_group: 'Group_Raw', col_sample: 'Sample_Name_Raw'}, inplace=True)
df_label['Group_Num'] = pd.to_numeric(df_label['Group_Raw'], errors='ignore')
df_c_raw = pd.read_csv(file_cluster)

In [214]:
df_c_raw=df_c_raw.set_index('Unnamed: 0')
df_c_raw['Class']=df_c_raw.index

In [215]:
import pandas as pd
import re

# ---------------------------------------------------------
# 1. 数据预处理函数
# ---------------------------------------------------------
def clean_sample_name(name):
    """
    统一格式规则：
    1. 去除末尾的 '.tsv'
    2. 去除开头的 'm' (将 mADT-1 变为 ADT-1)
    3. 去除首尾空格
    """
    if not isinstance(name, str):
        return name
    
    # 去除 .tsv
    cleaned = re.sub(r'\.tsv$', '', name.strip())
    
    # 去除开头的 'm' (仅当后面紧跟 ADT 时，防止误删其他以 m 开头的名字，虽然这里主要是 mADT)
    # 使用正则 ^m(?=ADT) 表示：如果开头是 m 且后面紧跟着 ADT，则删除 m
    cleaned = re.sub(r'^m(?=ADT)', '', cleaned)
    
    return cleaned


df_label['Match_Key'] = df_label['Sample_Name_Raw'].apply(clean_sample_name)


if 'Class' in df_c_raw.columns:
    source_series = df_c_raw['Class']
else:
    # 如果 Class 是索引
    source_series = df_c_raw.index

df_c_raw['Match_Key'] = source_series.apply(clean_sample_name)

df_merged = pd.merge(
    df_label, 
    df_c_raw, 
    on='Match_Key', 
    how='inner',
    suffixes=('_label', '_c_raw') # 给重复列名加后缀，避免冲突
)

df_final = df_merged.rename(columns={'Match_Key': 'Unified_Sample_Name'})


print("合并成功！重叠样本数量:", len(df_final))
print("\n前 10 行预览:")
print(df_final[['Unified_Sample_Name', 'Sample_Name_Raw', 'Class']].head(10))



合并成功！重叠样本数量: 50

前 10 行预览:
  Unified_Sample_Name Sample_Name_Raw        Class
0               ADT-1           ADT-1   mADT-1.tsv
1               ADT-2           ADT-2   mADT-2.tsv
2               ADT-3           ADT-3   mADT-3.tsv
3               ADT-4           ADT-4   mADT-4.tsv
4               ADT-5           ADT-5   mADT-5.tsv
5               ADT-6           ADT-6   mADT-6.tsv
6               ADT-7           ADT-7   mADT-7.tsv
7               ADT-8           ADT-8   mADT-8.tsv
8               ADT-9           ADT-9   mADT-9.tsv
9              ADT-10          ADT-10  mADT-10.tsv


In [216]:
# 定义映射关系
mapping = {0: 0, 1: 0, 2: 1}

# 应用映射
df_final['Group_Num'] = df_final['Group_Num'].map(mapping)

# 检查是否有未匹配到的值 (如果有 NaN 产生，说明原数据中有 0,1,2 以外的值)
if df_final['Group_Num'].isnull().any():
    print("警告：发现未定义的原始值，已转换为 NaN")
else:
    print("转换成功！")

# 查看转换后的分布
print(df_final['Group_Num'].value_counts().sort_index())

转换成功！
0    30
1    20
Name: Group_Num, dtype: int64


In [217]:
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np

# ---------------------------------------------------------
# 1. 构建列联表 (Crosstab)
# ---------------------------------------------------------
# 确保数据是整数或类别类型，去除可能的 NaN
df_clean = df_final.dropna(subset=['Group_Num', 'PhenoGraph'])

# 创建交叉表
contingency_table = pd.crosstab(df_clean['Group_Num'], df_clean['PhenoGraph'])

print("列联表 (Crosstab):")
print(contingency_table)
print("-" * 30)

row_sum = contingency_table.sum(axis=1)
col_sum = contingency_table.sum(axis=0)
total = contingency_table.values.sum()

expected = np.outer(row_sum, col_sum) / total
expected = pd.DataFrame(expected,
                        index=contingency_table.index,
                        columns=contingency_table.columns)
p_table = pd.DataFrame(index=contingency_table.index,
                       columns=contingency_table.columns)

for i in contingency_table.index:
    for j in contingency_table.columns:
        
        a = contingency_table.loc[i, j]
        b = contingency_table.loc[i].sum() - a
        c = contingency_table[j].sum() - a
        d = total - (a + b + c)
        
        table = [[a, b],
                 [c, d]]
        
        _, p = fisher_exact(table,alternative="greater")
        p_table.loc[i, j] = p

p_table = p_table.astype(float)
p_table.to_csv("SP_AD.csv")

列联表 (Crosstab):
PhenoGraph  SP_1  SP_2  SP_3  SP_4  SP_5
Group_Num                               
0              6    21     0     1     2
1              4    12     1     2     1
------------------------------


## SP and treatment

In [218]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 配置路径
# ==========================================
base_path = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC"
file_label = os.path.join(base_path, "Part 5 other figures in Fig1-4 Fig S1-6/anderson_adt.txt") # 请确认这个文件是否包含 0,1,2
file_cluster = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC/Part 3 discovery cohort TP SP TPL/dat_CN_percent_class.csv"
output_prefix = os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/Results_Merged_Fisher")
df_label = pd.read_csv(file_label, sep='\t', engine='python')
df_label.columns = [str(c).strip().lower() for c in df_label.columns]
col_group = df_label.columns[0] 
col_sample = df_label.columns[1]
df_label.rename(columns={col_group: 'Group_Raw', col_sample: 'Sample_Name_Raw'}, inplace=True)
df_label['Group_Num'] = pd.to_numeric(df_label['Group_Raw'], errors='ignore')
df_c_raw = pd.read_csv(file_cluster)
df_c_raw=df_c_raw.set_index('Unnamed: 0')
df_c_raw['Class']=df_c_raw.index
import numpy as np

# 确保 'Class' 列是字符串类型，防止报错
df_c_raw['Class'] = df_c_raw['Class'].astype(str)

# 逻辑：如果以 'mADT' 开头则为 0，否则 (即 'N' 开头) 为 1
df_c_raw['Group_Num'] = np.where(df_c_raw['Class'].str.startswith('mADT'), 0, 1)

print("新建列成功！前 5 行预览：")
print(df_c_raw[['Class', 'Group_Num']].head())

新建列成功！前 5 行预览：
                   Class  Group_Num
Unnamed: 0                         
mADT-1.tsv    mADT-1.tsv          0
mADT-10.tsv  mADT-10.tsv          0
mADT-11.tsv  mADT-11.tsv          0
mADT-12.tsv  mADT-12.tsv          0
mADT-13.tsv  mADT-13.tsv          0


In [219]:
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np
df_final=df_c_raw.copy()
# ---------------------------------------------------------
# 1. 构建列联表 (Crosstab)
# ---------------------------------------------------------
# 确保数据是整数或类别类型，去除可能的 NaN
df_clean = df_final.dropna(subset=['Group_Num', 'PhenoGraph'])

# 创建交叉表
contingency_table = pd.crosstab(df_clean['Group_Num'], df_clean['PhenoGraph'])

print("列联表 (Crosstab):")
print(contingency_table)
print("-" * 30)

row_sum = contingency_table.sum(axis=1)
col_sum = contingency_table.sum(axis=0)
total = contingency_table.values.sum()

expected = np.outer(row_sum, col_sum) / total
expected = pd.DataFrame(expected,
                        index=contingency_table.index,
                        columns=contingency_table.columns)
p_table = pd.DataFrame(index=contingency_table.index,
                       columns=contingency_table.columns)

for i in contingency_table.index:
    for j in contingency_table.columns:
        
        a = contingency_table.loc[i, j]
        b = contingency_table.loc[i].sum() - a
        c = contingency_table[j].sum() - a
        d = total - (a + b + c)
        
        table = [[a, b],
                 [c, d]]
        
        _, p = fisher_exact(table,alternative="greater")
        p_table.loc[i, j] = p

p_table = p_table.astype(float)
p_table.to_csv("SP_Treat.csv")

列联表 (Crosstab):
PhenoGraph  SP_1  SP_2  SP_3  SP_4  SP_5
Group_Num                               
0             10    33     1     3     3
1              1     4    15     6    30
------------------------------


In [220]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 配置路径
# ==========================================
base_path = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC"
file_label = os.path.join(base_path, "Part 5 other figures in Fig1-4 Fig S1-6/anderson_adt.txt") # 请确认这个文件是否包含 0,1,2
file_cluster = "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC/Part 3 discovery cohort TP SP TPL/dat_CN_percent_class.csv"
output_prefix = os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/Results_Merged_Fisher")
df_label = pd.read_csv(file_label, sep='\t', engine='python')
df_label.columns = [str(c).strip().lower() for c in df_label.columns]
col_group = df_label.columns[0] 
col_sample = df_label.columns[1]
df_label.rename(columns={col_group: 'Group_Raw', col_sample: 'Sample_Name_Raw'}, inplace=True)
df_label['Group_Num'] = pd.to_numeric(df_label['Group_Raw'], errors='ignore')
df_c_raw = pd.read_csv(file_cluster)

In [221]:
## gleason

In [222]:
gleason =  pd.read_csv('/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC/Part 5 other figures in Fig1-4 Fig S1-6/gleason.txt',sep='\t')

In [223]:
SP = pd.read_csv('/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC/Part 3 discovery cohort TP SP TPL/dat_CN_percent_class.csv', index_col=0)

In [224]:
Tumerpurity = pd.read_csv(os.path.join(base_path, "Part 3 discovery cohort TP SP TPL/mydata_pheno02.csv"), index_col=0)

In [225]:
TP = pd.read_csv(os.path.join(base_path, "/home/qyyuan/project/Lifei-Spatial/analysis/ChenleiLab_CODEX_HCC/Part 3 discovery cohort TP SP TPL/mydata_pheno_anno.csv"), index_col=0)

In [226]:
import pandas as pd

# ---------------------------------------------------------
# 步骤 1: 修改 gleason 的 Class 列
# 将 'N-1' 变为 'N-1.tsv'
# ---------------------------------------------------------

df_gleason = gleason.copy()

# 确保 Class 列是字符串类型，防止报错
df_gleason['Class'] = df_gleason['Class'].astype(str)

# 添加 '.tsv' 后缀
# 原始: 'N-3' -> 新: 'N-3.tsv'
df_gleason['Class'] = df_gleason['Class'] + '.tsv'

In [227]:

# ---------------------------------------------------------
# 步骤 2: 直接合并
# 以 gleason 的 'Class' 列 和 其他 df 的 索引 (Index) 进行合并
# 使用 inner join 只保留重叠的行
# ---------------------------------------------------------

# 为了方便 merge，先将其他 df 的索引重置为列，或者直接使用 left_on 和 right_index
# 方法：链式 merge

# 2.1 合并 Gleason 和 SP
# left_on='Class' (gleason的列), right_index=True (SP的行名)
merged_df = pd.merge(
    df_gleason, 
    SP['PhenoGraph'], 
    left_on='Class', 
    right_index=True, 
    how='inner'
)
# 重命名 SP 列以防冲突（虽然这里列名不同）
merged_df.rename(columns={0: 'PhenoGraph'} if 0 in merged_df.columns else {}, inplace=True)
# 如果 SP['PhenoGraph'] 是 Series，merge 后列名会自动保留为 'PhenoGraph' (如果是 named series) 或 0
# 上面 SP['PhenoGraph'] 如果是 Series，需要确保它有 name，或者 merge 后手动改名
if isinstance(SP['PhenoGraph'], pd.Series):
    if 'PhenoGraph' not in merged_df.columns:
        # 如果 merge 后列名是 0 或者其他，强制改为 PhenoGraph
        # 获取 merge 后产生的实际列名（排除 Class, Gleason）
        cols = [c for c in merged_df.columns if c not in ['Class', 'Gleason']]
        if len(cols) == 1:
            merged_df.rename(columns={cols[0]: 'PhenoGraph'}, inplace=True)

# 2.2 合并 Tumerpurity
merged_df = pd.merge(
    merged_df,
    Tumerpurity['TumorPurityLevel'],
    left_on='Class',
    right_index=True,
    how='inner'
)
# 修正列名
if isinstance(Tumerpurity['TumorPurityLevel'], pd.Series):
    cols = [c for c in merged_df.columns if c not in ['Class', 'Gleason', 'PhenoGraph']]
    # 找到刚才加入的那一列
    new_col = [c for c in cols if c not in merged_df.columns[:-1]][-1] # 简单逻辑：取最新加入的列
    # 更稳妥的方式：merge series 时，如果 series 有 name 则用 name，否则用 0。
    # 这里我们直接根据类型判断重命名
    if 'TumorPurityLevel' not in merged_df.columns:
         # 寻找非 Class, Gleason, PhenoGraph 的列
         target_col = [c for c in merged_df.columns if c not in ['Class', 'Gleason', 'PhenoGraph'] and c != 'TumorPurityLevel'][-1]
         merged_df.rename(columns={target_col: 'TumorPurityLevel'}, inplace=True)

# 2.3 合并 TP (Cluster)
merged_df = pd.merge(
    merged_df,
    TP['Cluster'],
    left_on='Class',
    right_index=True,
    how='inner'
)
# 修正列名
if isinstance(TP['Cluster'], pd.Series):
    if 'Cluster' not in merged_df.columns:
        target_col = [c for c in merged_df.columns if c not in ['Class', 'Gleason', 'PhenoGraph', 'TumorPurityLevel']][-1]
        merged_df.rename(columns={target_col: 'Cluster'}, inplace=True)

# ---------------------------------------------------------
# 最终整理
# ---------------------------------------------------------

# 1. 将 'Class' 列设为索引 (还原为您想要的行名状态)
final_df = merged_df.set_index('Class')

# 2. 确保列顺序和名称正确
required_columns = ['Gleason', 'PhenoGraph', 'TumorPurityLevel', 'Cluster']

# 检查是否有列缺失并报错提示
missing = [c for c in required_columns if c not in final_df.columns]
if missing:
    print(f"警告：合并后缺少以下列：{missing}")
else:
    final_df = final_df[required_columns]

print("\n合并成功！")
print(f"重叠样本数量：{len(final_df)}")
print("\n结果预览 (Index 为统一后的文件名):")
print(final_df.head())

# df_final 即为最终结果
df_final = final_df


合并成功！
重叠样本数量：53

结果预览 (Index 为统一后的文件名):
         Gleason PhenoGraph TumorPurityLevel  Cluster
Class                                                
N-3.tsv        6       SP_5                H    TP_12
N-4.tsv        6       SP_5                H    TP_14
N-5.tsv        6       SP_5                H    TP_12
N-6.tsv        6       SP_3                L  TP_4-13
N-7.tsv        6       SP_3                L  TP_4-13


In [228]:
# ❌ 错误的写法 (您之前的代码)
# df_final[df_final['Gleason']<=7]['Gleason'] = 0  <-- 这可能只修改了临时副本
# df_final[df_final['Gleason']>7]['Gleason'] = 1

# ✅ 正确的写法 (使用 .loc)
# 逻辑：在 df_final 中，找到 Gleason <= 7 的行，将这些行的 'Gleason' 列设为 0
df_final.loc[df_final['Gleason'] <= 7, 'Gleason'] = 0

# 逻辑：在 df_final 中，找到 Gleason > 7 的行，将这些行的 'Gleason' 列设为 1
df_final.loc[df_final['Gleason'] > 7, 'Gleason'] = 1

# 验证修改结果
print("修改后的 Gleason 分布:")
print(df_final['Gleason'].value_counts())
print("\n前几行预览:")
print(df_final.head())

修改后的 Gleason 分布:
1    27
0    26
Name: Gleason, dtype: int64

前几行预览:
         Gleason PhenoGraph TumorPurityLevel  Cluster
Class                                                
N-3.tsv        0       SP_5                H    TP_12
N-4.tsv        0       SP_5                H    TP_14
N-5.tsv        0       SP_5                H    TP_12
N-6.tsv        0       SP_3                L  TP_4-13
N-7.tsv        0       SP_3                L  TP_4-13


In [229]:
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np

# ---------------------------------------------------------
# 1. 构建列联表 (Crosstab)
# ---------------------------------------------------------
# 确保数据是整数或类别类型，去除可能的 NaN
df_clean = df_final.dropna(subset=['Gleason', 'PhenoGraph'])

# 创建交叉表
contingency_table = pd.crosstab(df_clean['Gleason'], df_clean['PhenoGraph'])

print("列联表 (Crosstab):")
print(contingency_table)
print("-" * 30)

row_sum = contingency_table.sum(axis=1)
col_sum = contingency_table.sum(axis=0)
total = contingency_table.values.sum()

expected = np.outer(row_sum, col_sum) / total
expected = pd.DataFrame(expected,
                        index=contingency_table.index,
                        columns=contingency_table.columns)
p_table = pd.DataFrame(index=contingency_table.index,
                       columns=contingency_table.columns)

for i in contingency_table.index:
    for j in contingency_table.columns:
        
        a = contingency_table.loc[i, j]
        b = contingency_table.loc[i].sum() - a
        c = contingency_table[j].sum() - a
        d = total - (a + b + c)
        
        table = [[a, b],
                 [c, d]]
        
        _, p = fisher_exact(table,alternative="greater")
        p_table.loc[i, j] = p

p_table = p_table.astype(float)
p_table.to_csv("SP_GL.csv")

列联表 (Crosstab):
PhenoGraph  SP_1  SP_2  SP_3  SP_4  SP_5
Gleason                                 
0              0     4     9     2    11
1              1     0     5     4    17
------------------------------


In [230]:
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np

# ---------------------------------------------------------
# 1. 构建列联表 (Crosstab)
# ---------------------------------------------------------
# 确保数据是整数或类别类型，去除可能的 NaN
df_clean = df_final.dropna(subset=['Gleason', 'TumorPurityLevel'])

# 创建交叉表
contingency_table = pd.crosstab(df_clean['Gleason'], df_clean['TumorPurityLevel'])

print("列联表 (Crosstab):")
print(contingency_table)
print("-" * 30)

row_sum = contingency_table.sum(axis=1)
col_sum = contingency_table.sum(axis=0)
total = contingency_table.values.sum()

expected = np.outer(row_sum, col_sum) / total
expected = pd.DataFrame(expected,
                        index=contingency_table.index,
                        columns=contingency_table.columns)
p_table = pd.DataFrame(index=contingency_table.index,
                       columns=contingency_table.columns)

for i in contingency_table.index:
    for j in contingency_table.columns:
        
        a = contingency_table.loc[i, j]
        b = contingency_table.loc[i].sum() - a
        c = contingency_table[j].sum() - a
        d = total - (a + b + c)
        
        table = [[a, b],
                 [c, d]]
        
        _, p = fisher_exact(table,alternative="greater")
        p_table.loc[i, j] = p

p_table = p_table.astype(float)
p_table.to_csv("TumorPurityLevel_GL.csv")

列联表 (Crosstab):
TumorPurityLevel   H  L   M
Gleason                    
0                  4  4  18
1                 20  5   2
------------------------------


In [231]:
import pandas as pd
from scipy.stats import fisher_exact
import numpy as np

# ---------------------------------------------------------
# 1. 构建列联表 (Crosstab)
# ---------------------------------------------------------
# 确保数据是整数或类别类型，去除可能的 NaN
df_clean = df_final.dropna(subset=['Gleason', 'Cluster'])

# 创建交叉表
contingency_table = pd.crosstab(df_clean['Gleason'], df_clean['Cluster'])

print("列联表 (Crosstab):")
print(contingency_table)
print("-" * 30)

row_sum = contingency_table.sum(axis=1)
col_sum = contingency_table.sum(axis=0)
total = contingency_table.values.sum()

expected = np.outer(row_sum, col_sum) / total
expected = pd.DataFrame(expected,
                        index=contingency_table.index,
                        columns=contingency_table.columns)
p_table = pd.DataFrame(index=contingency_table.index,
                       columns=contingency_table.columns)

for i in contingency_table.index:
    for j in contingency_table.columns:
        
        a = contingency_table.loc[i, j]
        b = contingency_table.loc[i].sum() - a
        c = contingency_table[j].sum() - a
        d = total - (a + b + c)
        
        table = [[a, b],
                 [c, d]]
        
        _, p = fisher_exact(table,alternative="greater")
        p_table.loc[i, j] = p

p_table = p_table.astype(float)
p_table.to_csv("TP_GL.csv")

列联表 (Crosstab):
Cluster  TP_10  TP_12  TP_14  TP_15  TP_16  TP_4-13  TP_5  TP_6  TP_9
Gleason                                                              
0            0      8      3      0      0       10     2     3     0
1            1     11      2      1      2        5     4     0     1
------------------------------
